# Data cleaning & manipulation

## **Notebook Overview**

This notebook prepares the dataset for a regression model that predicts duration_difference_min.
The workflow is structured into modular functions, each handling a specific step:
Data type fixing and standardization
Missing values handling
Outlier correction and filtering
Feature engineering
Categorical encoding
After preprocessing, irrelevant and leakage-prone columns are removed.
Feature selection is performed using multiple regression-based methods, and results are combined into a summary table.
Finally, XGBoost models are trained on different feature subsets and compared using RMSE, MAE, and R².

## Libraries

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from category_encoders import TargetEncoder
import sys
from pathlib import Path
sys.path.append(str(Path("..").resolve()))
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.pyplot as plt
import seaborn as sns
import math
from src.data_cleaning_and_manipulations import fix_data_types, handle_missing_values, handle_outliers, add_features
from src.data_cleaning_and_manipulations import add_probability_features, encode_categorical_columns
from src.data_cleaning_and_manipulations import drop_unnecessary_columns, run_feature_selection_methods_classification
from src.data_cleaning_and_manipulations import test_xgboost_by_feature_votes_split, fill_by_ref_group_median, add_probability_features

## Imports

In [5]:
train_df = pd.read_csv(r'../data/model_datasets/train_df.csv', encoding='utf-8-sig')
print(f"Train:      {len(train_df):,} rows")

Train:      63,616 rows


## 1. Manage Data types

**Step 1: Fix data types and standardize columns**
- Convert datetime columns to string format
- Rename 'hour_rounded' to 'full_hour' for consistency
- Convert 'day' to an ordered categorical variable (Sunday → Saturday)
- Ensure key categorical columns are treated as strings
- Clean text fields (e.g., remove whitespace from 'route_type')

In [6]:
train_df = fix_data_types(train_df)

## 2. Missing values handling

**Handle Missing Values**

- Impute missing values using hierarchical group-based strategies (from granular to general levels)
- Apply fallback imputation using global statistics from a reference dataset (train set)
- Handle categorical missing values by assigning default labels (e.g., "unknown")
- Fill binary/indicator features with default values where appropriate
- Ensure consistency in numerical features using group-level and global aggregations
- Correct invalid or inconsistent values (e.g., zero or missing lengths)
- Optionally print missing values summary before and after processing

In [7]:
train_df = handle_missing_values(train_df, printing_missing_values=True)

Missing values BEFORE handling:
Total_Passengers           813
Avg_Passengers_Per_Bus     813
length_in_buffer_m         119
curvity                    119
circular_route             119
duration_min_actual         89
duration_difference_min     89
speed_kmh_actual            89
route_length                 4
dtype: int64
--------------------------------------------------
Missing values AFTER handling:
Series([], dtype: int64)
--------------------------------------------------


## 3. Outliers handling

**Handle Outliers**

- Identify and correct unrealistic values in key features
- Adjust planned trips that cross midnight by recalculating duration and speed
- Remove extreme values in the target variable based on defined thresholds
- Report number and proportion of removed observations
- Optionally visualize feature distributions before and after outlier handling using boxplots

In [8]:
train_df = handle_outliers(train_df, boxplots=False, boxplot_cols=None, verbose=False, remove_target_outliers=True)

## 4. Add Features

**Add Features**

- Create peak-hour indicator based on predefined morning and afternoon peak periods
- Create urban/interurban indicator based on route length
- Calculate the share of the route located within the public transport buffer
- Create peak-weighted route-buffer feature
- Add interaction features between passengers, stops, and peak-hour conditions
- Extract planned departure and arrival hour features
- Create combined categorical route identifier using agency, line, direction, and alternative values

In [9]:
train_df = add_features(train_df)

### 4.2. Early/On_time probabilities 

In [7]:
train_df = add_probability_features(train_df)
val_df = add_probability_features(
    val_df,
    ref_df=train_df
)

## 5. Categorical Data Handling

**Encode Categorical Columns**

- Remove duplicate columns before and after the encoding process
- Convert day into an ordinal numerical feature based on weekday order
- Apply target encoding to high-cardinality categorical features
- Fit the target encoder on the training dataset only
- Apply the same encoder to validation and test datasets to avoid data leakage
- Return the transformed dataset along with the fitted encoder for reuse

In [10]:
train_df, te = encode_categorical_columns(train_df)

In [4]:
train_df, te = manipulate_df_process(
    train_df,
    train=True
)

val_df = manipulate_df_process(
    val_df,
    ref_df=train_df,
    train=False,
    te=te
)

train_df = drop_unnecessary_columns(train_df)
val_df = drop_unnecessary_columns(val_df)

## 6. Feature Selection

**Feature Selection Process** 

-  Drop columns that are not suitable for modeling:
    - Original categorical columns that were already encoded
    - Raw text and datetime columns that cannot be used directly by the model
    - Duplicate or redundant columns
    - Data leakage columns, such as actual duration and actual speed, that directly reveal  information related to the target
- Define the target variable: duration_difference_min
- Create X_train and y_train after removing the target column
- Run several feature selection methods for a regression problem:
    - Random Forest feature importance
    - Gradient Boosting feature importance
    - Lasso coefficient selection
    - Linear SVR coefficient selection
    - Ridge coefficient ranking
-  Store the result of each method in a feature-selection summary table
-  Calculate a final Sum score for each feature, based on how many methods selected it
-  Compare XGBoost model performance using different feature sets:
    - All available features
    - Features selected by all methods (Sum == 5)
    - Features selected by at least four methods (Sum >= 4)
-  Evaluate and compare the results using RMSE, MAE, and R².

In [11]:
## Drop unnecessary columns
train_df = drop_unnecessary_columns(train_df)

# Define target
target_col = "delay_cat"

# Define X and y
X = train_df.drop(
    columns=[target_col, "duration_difference_min"],
    errors="ignore"
)

y = train_df[target_col]

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Run feature selection ONLY on train
selection_df = run_feature_selection_methods_classification(
    X_train,
    y_train
)

selection_df.head()

Feature selection methods to be used:
1. RandomForest
2. GradientBoost
3. LinearSVC
4. LogisticRegression_L2
--------------------------------------------------
Finished RandomForest (1/4)
Finished GradientBoost (2/4)
Finished LinearSVC (3/4)
Finished LogisticRegression_L2 (4/4)


,Feature,LinearSVC,GradientBoost,RandomForest,Logistic_L2,Sum
0,full_hour,1,1,1,1,4
1,route_id,1,1,1,1,4
2,destination_station_encoded,1,1,1,1,4
3,origin_station_encoded,1,1,1,1,4
4,destination_city_encoded,1,1,1,1,4


In [12]:
results_df, models = test_xgboost_by_feature_votes_split(
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test,
    selection_df=selection_df,
    vote_options=[4, 3]
)

results_df

Features with at least 4 votes
Number of features: 28
              precision    recall  f1-score   support

       delay       0.87      0.93      0.90      8924
       early       0.72      0.33      0.46       678
     on_time       0.72      0.65      0.68      3071

    accuracy                           0.83     12673
   macro avg       0.77      0.64      0.68     12673
weighted avg       0.82      0.83      0.82     12673

Features with at least 3 votes
Number of features: 30
              precision    recall  f1-score   support

       delay       0.87      0.93      0.90      8924
       early       0.72      0.33      0.46       678
     on_time       0.71      0.65      0.68      3071

    accuracy                           0.83     12673
   macro avg       0.77      0.64      0.68     12673
weighted avg       0.82      0.83      0.82     12673



,min_votes,n_features,accuracy,macro_f1,weighted_f1
0,4,28,0.832084,0.679194,0.823292
1,3,30,0.831374,0.678814,0.822677


In [ ]:
### No reason to reduce features

## 7. Manipulation pipelines

Two function will be used before modelling to make the manipulation and cleaniogn process and ensure that no data leakage happens:
- manipulate_df_process
- drop_unnecessary_columns

Both of the functions aure in /src/data_cleaning_and_manipulations.py